# 03 Linear SVM：用企鹅数据理解“画一条尽量稳的分界线”

Linear SVM 最适合用图来理解。为了让分界线清楚，我们只挑两种企鹅：

- Adelie
- Chinstrap

再只用两个指标：

- `bill_length_mm`：嘴巴长度
- `bill_depth_mm`：嘴巴深度

任务是：根据嘴巴长度和深度，预测企鹅是 Adelie 还是 Chinstrap。


## 运行前说明

这些 notebook 是教学演示，不是医学诊断或生物学结论。我们会使用你放在项目里的真实表格数据，但这里只是学习机器学习模型怎么工作。

数据路径：

- 心脏病数据：`../data/ml_data/heart_data.csv`
- 企鹅数据：`../data/ml_data/penguins.csv`

推荐在本项目已有的 Docker/Jupyter 环境里运行，因为 `environment/Dockerfile` 已经安装了 `numpy`、`pandas`、`scikit-learn`、`matplotlib`、`seaborn`。

如果你想在本机 Python 里运行，可以先安装：

```bash
pip install numpy pandas scikit-learn matplotlib seaborn notebook
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. 读取并筛选数据

我们先读入企鹅数据，然后只保留 Adelie 和 Chinstrap。这样就是一个二分类问题，更容易画出一条线。


In [ ]:
data_path = Path('../data/ml_data/penguins.csv')
penguins = pd.read_csv(data_path)

if '' in penguins.columns:
    penguins = penguins.drop(columns=[''])

features = ['bill_length_mm', 'bill_depth_mm']
target = 'species'

svm_data = penguins[penguins['species'].isin(['Adelie', 'Chinstrap'])].dropna(subset=features + [target]).copy()

print('数据形状：', svm_data.shape)
print(svm_data['species'].value_counts())
svm_data[features + [target]].head()


## 2. 先把两种企鹅画出来

每个点是一只企鹅。横轴是嘴巴长度，纵轴是嘴巴深度。

Linear SVM 要做的事就是：在图上找一条直线，尽量把两种颜色分开。


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=svm_data,
    x='bill_length_mm',
    y='bill_depth_mm',
    hue='species',
    palette=['#2b7bba', '#d95f02'],
    s=80,
)
plt.title('Adelie 和 Chinstrap 的嘴巴长度/深度')
plt.xlabel('嘴巴长度 bill_length_mm')
plt.ylabel('嘴巴深度 bill_depth_mm')
plt.show()


## 3. Linear SVM 的核心直觉

如果只是“画一条能分开的线”，可能有很多画法。

SVM 更挑剔：它想找一条线，让这条线离两边最近的点都尽量远。这个距离叫 **margin（间隔）**。

离线最近、最影响画线位置的点叫 **support vectors（支持向量）**。


In [ ]:
X = svm_data[features]
y = svm_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

model = make_pipeline(
    StandardScaler(),
    SVC(kernel='linear', C=1.0, probability=True, random_state=RANDOM_STATE)
)

model.fit(X_train, y_train)
print('训练完成')
print('测试集准确率：', round(model.score(X_test, y_test), 3))


## 4. 预测一只新企鹅

输入嘴巴长度和嘴巴深度，模型会判断更像 Adelie 还是 Chinstrap。


In [ ]:
new_penguin = pd.DataFrame({
    'bill_length_mm': [45.0],
    'bill_depth_mm': [18.5],
})

predicted_species = model.predict(new_penguin)[0]
probabilities = pd.Series(model.predict_proba(new_penguin)[0], index=model.classes_).sort_values(ascending=False)

display(new_penguin)
print('预测企鹅种类：', predicted_species)
print('每个种类的预测概率：')
display(probabilities.to_frame('probability'))


## 5. 评估模型

我们仍然用准确率、分类报告、混淆矩阵来看表现。


In [ ]:
y_pred = model.predict(X_test)

print('准确率：', round(accuracy_score(y_test, y_pred), 3))
print()
print('分类报告：')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap='Purples')
plt.title('Linear SVM 企鹅分类混淆矩阵')
plt.show()


## 6. 画出分界线、margin 和支持向量

下面的图是 Linear SVM 最重要的一张图：

- 黑色实线：最终分界线
- 黑色虚线：margin 边界
- 空心大圆：支持向量


In [ ]:
def plot_linear_svm(model, X, y):
    scaler = model.named_steps['standardscaler']
    svm = model.named_steps['svc']

    x_min, x_max = X.iloc[:, 0].min() - 1.5, X.iloc[:, 0].max() + 1.5
    y_min, y_max = X.iloc[:, 1].min() - 1.0, X.iloc[:, 1].max() + 1.0
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    grid = pd.DataFrame({
        X.columns[0]: xx.ravel(),
        X.columns[1]: yy.ravel(),
    })
    decision = model.decision_function(grid).reshape(xx.shape)
    support_vectors_original = scaler.inverse_transform(svm.support_vectors_)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, decision, levels=20, cmap='coolwarm', alpha=0.3)
    plt.contour(xx, yy, decision, levels=[-1, 0, 1], colors='black', linestyles=['--', '-', '--'], linewidths=[1.3, 2.3, 1.3])
    sns.scatterplot(x=X.iloc[:, 0], y=X.iloc[:, 1], hue=y, palette=['#2b7bba', '#d95f02'], s=75, edgecolor='white')
    plt.scatter(
        support_vectors_original[:, 0],
        support_vectors_original[:, 1],
        s=150,
        facecolors='none',
        edgecolors='black',
        linewidths=1.5,
        label='支持向量'
    )
    plt.title('Linear SVM 的分界线、margin 和支持向量')
    plt.xlabel(X.columns[0])
    plt.ylabel(X.columns[1])
    plt.legend(title='species')
    plt.show()

plot_linear_svm(model, X, y)


## 7. 小结

Linear SVM 适合：

- 两类数据大致可以用一条直线分开。
- 想理解“分界线”和“离边界最近的关键样本”。
- 特征很多时，也常常能作为一个稳的基础模型。

在企鹅例子里，它像是在图上画一条线，把 Adelie 和 Chinstrap 尽量稳稳分开。
